# Import modules

In [1]:
import os
import sys
import requests                      # HTTP client for API calls
import pandas as pd                  # Tabular data handling
from datetime import datetime        # Datetime handling
from typing import Iterable, Optional, Dict, Union
import matplotlib.pyplot as plt
import yfinance as yf
from pprint import pprint as pp

# Ensure repo root is on PYTHONPATH (CI safety)

In [2]:
REPO_ROOT = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(REPO_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)


# Secrets validation (FAIL FAST)

In [3]:
REQUIRED_ENV_VARS = [
    "EMAIL_USER",
    "EMAIL_SENDER",
    "EMAIL_SENDER_PSW",
]

missing = [v for v in REQUIRED_ENV_VARS if not os.getenv(v)]
if missing:
    raise RuntimeError(
        f"Missing required environment variables: {', '.join(missing)}"
    )

email_user = os.getenv("EMAIL_USER")
email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
email_sender = os.getenv("EMAIL_SENDER")

# Import internal modules

In [4]:
from src.debug_print import debug_print
from src.fetch_lse_tickers import get_ftse100
from src.exchange_rates import get_share_prices_2 as share_prices
from src.exchange_rates_v2 import get_share_prices_2_with_fundamentals
from src.roi_hit_v2 import get_first_roi_hit
from src.plot_shares_ROI import plot_candles_volatility_volume_roi as ROI
from src.extract_latest_fundamentals import extract_latest_fundamentals
from src.detect_undervalued import detect_undervalued
from src.utils.email_sender import (
    send_email_html_multi_inline_images,
    build_roi_email_with_action_images,
)
from src.utils.email_undervalued_shares import build_undervalued_shares_email_with_images

# Set up variables

In [5]:
base_currency = "GBP"
target_currencies = ["USD", "GBP", "EUR", "JPY"]
cryptos = ["BTC", "ETH"]
shares = ['FRES','ENT','GLEN','MNG','PHNX','VOD']
email_recipients = ["ingcarldan@gmail.com"]

start_date = pd.Timestamp(2024,1,1)
purchase_date = pd.Timestamp(2026,1,2)
end_date = pd.Timestamp.today().normalize()
ROI_target = 0.135

email_user = os.getenv("EMAIL_USER")
email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
email_sender = os.getenv("EMAIL_SENDER")


# Ensure output directory exists (CI requirement)

In [6]:
pics_dir = os.path.join(os.getcwd(), "output")
os.makedirs(pics_dir, exist_ok=True)


# Get TOP 100 shares from FTSE

In [7]:
ftse100 = get_ftse100()
ftse100["Yahoo_Ticker"] = ftse100["Ticker"] + ".L"
shares_lse = ftse100["Yahoo_Ticker"].to_list()

[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None get_ftse100 succeeded


# SHARES PRICES WITH INFO

In [8]:
df_shares_fund, failed_tickers_list = get_share_prices_2_with_fundamentals(
    tickers=shares_lse,
    start=start_date,
    end=end_date,
    base_currency = base_currency,
    vol_window = 20,
    
)

actions_list   = df_shares_fund.columns.get_level_values("ACTION").unique().to_list()
currencies_list = df_shares_fund.columns.get_level_values("CURRENCY").unique()
metrics   = df_shares_fund.columns.get_level_values("METRIC").unique()

# EXTRACT UNDERVALUED SHARES

In [9]:
df_fund = extract_latest_fundamentals(
    df=df_shares_fund,
    evaluation_date=purchase_date,
)

undervalued_shares = detect_undervalued(df_fund)
filt = undervalued_shares[undervalued_shares["UndervaluedScore"] > 0]
undervalued_shares_list = filt.index.to_list()
filt["Ticker"] = filt.index.str.split(".").str[0]

filt["Company"] = (
    filt["Ticker"]
    .map(ftse100.set_index("Ticker")["Company"])
)
currency_convertion = [s.split('_')[1] for s in filt.index.to_list()]

filt['original currency'] = [s.split('→')[0] for s in currency_convertion]
filt['converted currency'] = [s.split('→')[1] for s in currency_convertion]
filt = filt[['Company', 'Ticker', 'UndervaluedScore', 'original currency', 'converted currency', 
                'Price', 'EPS', 'BookValue', 'Dividend', 'P/E', 'P/B']]
ROI(
    df=df_shares_fund,
    actions=filt.index.to_list(),
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_date= purchase_date,
    roi_target=ROI_target
)

/tmp/ipykernel_2286/3051297194.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt["Ticker"] = filt.index.str.split(".").str[0]
/tmp/ipykernel_2286/3051297194.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt["Company"] = (
/tmp/ipykernel_2286/3051297194.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_

[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None plot_candles_volatility_volume_roi succeeded


# SEND EMAIL FOR UNDERVALUED SHARES

In [10]:
try:
    text_body, html_body, inline_images = build_undervalued_shares_email_with_images(
        df_undervalued_shares=filt,
        image_dir='output'
    )

except Exception as e:
    print(f"ERROR in build_undervalued_shares_email_with_images {type(e).__name__}: {e}")
try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - Undevalued shares – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=text_body,
        html_body=html_body,
        inline_images=inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")
        


[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/email_undervalued_shares.py
   Function: build_undervalued_shares_email_with_images
   Line: 64
None image_dir: /home/runner/work/finance/finance/notebooks/output


[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None send_email_html_multi_inline_images succeeded


# ANALYZE SHARES' PORTFOLIO to get ROI Dates

In [11]:
portfolio = {}
SHARES_FULL_LIST = [s + '.L_GBp→GBP' if not s.endswith('.L_GBp→GBP') else s for s in shares]

for action in SHARES_FULL_LIST:
    try:
        action_clean = action.split(".L")[0]
        portfolio[action_clean] = get_first_roi_hit(
        df=df_shares_fund,
        action=action,
        purchase_date=purchase_date,
        roi_target=ROI_target
    )
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

#  PLOT SHARE - plot_candles_volatility_volume_roi 


In [12]:
SHARES_FULL_LIST = [
    s + ".L_GBP→GBP" if not s.endswith(".L_GBP→GBP") else s
    for s in shares
]

actions = df_shares_fund.columns.get_level_values("ACTION").unique()
mask = actions.str.contains("|".join(shares), case=False, regex=True)
filtered_actions = actions[mask].to_list()

ROI(
    df=df_shares_fund,
    actions=filtered_actions,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_date=purchase_date,
    roi_target=ROI_target,
)

[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None plot_candles_volatility_volume_roi succeeded


# Build email content

In [13]:
pics_dir = os.path.join(os.getcwd(),"output")

try:
    text_body, html_body, inline_images = build_roi_email_with_action_images(
        roi_data=portfolio, #df_fres,
        image_dir=pics_dir,  # contains AAL.png, VOD.png, ...
    )
except Exception as e:
    print(f"Could not run build_roi_email_with_action_images {type(e).__name__}: {e}")


[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/email_sender.py
   Function: build_roi_email_with_action_images
   Line: 422
None→ Converted data['PURCHASE DATE'] from str to <class 'pandas._libs.tslibs.timestamps.Timestamp'> 2026-01-02 00:00:00


# Send email (TLS via your utility)

In [14]:

try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - ROI targets – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=text_body,
        html_body=html_body,
        inline_images=inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")

[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None send_email_html_multi_inline_images succeeded


# PLOT PURCHASED SHARES

In [15]:
# check if shares name ends with .L_GBP→GBP
for i, s in enumerate(shares):
    if not s.endswith('.L_GBp→GBP'):
        shares[i] = s + '.L_GBp→GBP'

ROI(
    df=df_shares_fund,
    actions=shares,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_date= purchase_date,
    roi_target=ROI_target
)

[DEBUG PRINT]
   File: /home/runner/work/finance/finance/src/utils/retry_decorator.py
   Function: wrapper
   Line: 38
None plot_candles_volatility_volume_roi succeeded
